# Bronze Employee Ingestion

This notebook demonstrates a simple bronze-layer ingestion flow for employee data from multiple source formats.

* Validate compute and storage access
* Read employee data from CSV, JSON, and XML files
* Perform basic profiling such as schema, row count, and null checks
* Write raw data into bronze Delta locations
* Archive source files after ingestion


In [0]:
# Verify that the Databricks compute is responding before starting ingestion.
print("Hello! My Databricks compute is working.")

In [0]:
# Define the source file path for the employee CSV file in the landing container.
landing_path = "abfss://landing@bronzeadlsstorage.dfs.core.windows.net/csv/Employee.csv"

In [0]:
# List the files available in the CSV landing directory before reading data.
landing_path = (
    "abfss://landing@bronzeadlsstorage.dfs.core.windows.net/csv/"
)

display(dbutils.fs.ls(landing_path))

In [0]:
# Read the employee CSV file into a Spark DataFrame and preview the data.
employee_df = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(
        "abfss://landing@bronzeadlsstorage.dfs.core.windows.net/csv/Employee.csv"
    )
)

display(employee_df)

In [0]:
# Inspect the inferred schema of the employee CSV dataset.
employee_df.printSchema()

In [0]:
# Count the total number of employee records loaded from the CSV file.
employee_df.count()

In [0]:
# Check null values across all columns in the employee CSV dataset.
from pyspark.sql.functions import col, count, when

display(
    employee_df.select(
        [count(when(col(c).isNull(), c)).alias(c) for c in employee_df.columns]
    )
)

In [0]:
# Write the employee CSV data to the bronze Delta location.
employee_df.write \
.mode("overwrite") \
.format("delta") \
.save("abfss://landing@bronzeadlsstorage.dfs.core.windows.net/bronze/employee/")

In [0]:
# Read the bronze employee Delta data to verify the write operation.
bronze_df = spark.read.format("delta").load(
    "abfss://landing@bronzeadlsstorage.dfs.core.windows.net/bronze/employee/"
)

display(bronze_df)

In [0]:
# Move the processed employee CSV file from landing to the archive container.
dbutils.fs.mv(
    "abfss://landing@bronzeadlsstorage.dfs.core.windows.net/csv/Employee.csv",
    "abfss://archive@bronzeadlsstorage.dfs.core.windows.net/archive/Employee.csv"
)

## CSV Employee Ingestion

This section ingests employee data from a CSV file, performs basic validation, stores it in the bronze layer, and archives the source file.


## JSON Employee Ingestion

This section reads employee data from a JSON file, profiles the dataset, writes it to the bronze Delta location, and archives the source file.


## XML Employee Ingestion

This section processes employee data from an XML file, validates the schema and nulls, writes it into bronze storage, and then archives the original file.


In [0]:
# List archived files to confirm the employee CSV file was moved successfully.
display(
    dbutils.fs.ls(
        "abfss://archive@bronzeadlsstorage.dfs.core.windows.net/archive/"
    )
)

In [0]:
# Read the employee JSON file into a Spark DataFrame and preview the records.
EMPJSON_df = (
    spark.read
    .json(
        "abfss://archive@bronzeadlsstorage.dfs.core.windows.net/archive/EMPJSON.json"
    )
)

display(EMPJSON_df)

In [0]:
# Inspect the schema of the employee JSON dataset.
EMPJSON_df.printSchema()

In [0]:
# Count the total number of employee records loaded from the JSON file.
EMPJSON_df.count()

In [0]:
# Check null values across all columns in the employee JSON dataset.
from pyspark.sql.functions import col, count, when

display(
    EMPJSON_df.select(
        [count(when(col(c).isNull(), c)).alias(c) for c in EMPJSON_df.columns]
    )
)

In [0]:
# Write the employee JSON data to the bronze Delta location.
EMPJSON_df.write \
.mode("overwrite") \
.format("delta") \
.save("abfss://landing@bronzeadlsstorage.dfs.core.windows.net/bronze/EMPJSON/")

In [0]:
# Read the bronze JSON Delta data to confirm the write completed successfully.
bronze_df = spark.read.format("delta").load(
    "abfss://landing@bronzeadlsstorage.dfs.core.windows.net/bronze/EMPJSON/"
)

display(bronze_df)

In [0]:
# Move the processed employee JSON file from landing to the archive container.
dbutils.fs.mv(
    "abfss://landing@bronzeadlsstorage.dfs.core.windows.net/json/EMPJSON.json",
    "abfss://archive@bronzeadlsstorage.dfs.core.windows.net/archive/EMPJSON.json"
)

In [0]:
# Read the employee XML file into a Spark DataFrame and preview the records.
EMPXML_df = (
    spark.read
    .format("xml")
    .option("rowTag", "record")
    .load(
        "abfss://landing@bronzeadlsstorage.dfs.core.windows.net/xml/EMPXML.xml"
    )
)

display(EMPXML_df)

In [0]:
# Inspect the schema of the employee XML dataset.
EMPXML_df.printSchema()

In [0]:
# Count the total number of employee records loaded from the XML file.
EMPXML_df.count()

In [0]:
# Check null values across all columns in the employee XML dataset.
from pyspark.sql.functions import col, count, when

display(
    EMPXML_df.select(
        [count(when(col(c).isNull(), c)).alias(c) for c in EMPXML_df.columns]
    )
)

In [0]:
# Write the employee XML data to the bronze Delta location with schema overwrite enabled.
EMPXML_df.write \
.mode("overwrite") \
.format("delta") \
.option("overwriteSchema", "true") \
.save("abfss://landing@bronzeadlsstorage.dfs.core.windows.net/bronze/EMPXML/")

In [0]:
# Read the bronze XML Delta data to validate the write operation.
bronze_df = spark.read.format("delta").load(
    "abfss://landing@bronzeadlsstorage.dfs.core.windows.net/bronze/EMPXML/"
)

display(bronze_df)

In [0]:
# Move the processed employee XML file from landing to the archive container.
dbutils.fs.mv(
    "abfss://landing@bronzeadlsstorage.dfs.core.windows.net/xml/EMPXML.xml",
    "abfss://archive@bronzeadlsstorage.dfs.core.windows.net/archive/EMPXML.xml"
)